# Manual checks for `clean_human_survey.py`

In [ ]:
import sys
sys.path.insert(0, ".")

import pandas as pd

import attention_registry as ar
from clean_human_survey import (
    build_rename_map,
    clean_survey,
    describe_structure,
    detect_structure,
    load_qualtrics,
    split_outputs,
)

FIXTURE = "clean_survey_CHECKS_input.csv" # -- this is just brother_utility1_deontology1_choice1 survey output from Qualtrics
SURVEY = {"scenario": "brother", "util": 1, "deon": 1, "choice": 1}

raw, codebook = load_qualtrics(FIXTURE)
structure = detect_structure(codebook)
bank = ar.parse_outcome_bank()
answers = ar.build_answers(SURVEY, structure, bank)

print(f"Raw responses in file: {len(raw)}")

Raw responses in file: 27


## Check 1

Run the real cleaning pipeline, then confirm `kept + excluded == raw`. 

In [2]:
result = clean_survey(FIXTURE, answers, min_attention_correct=7)

n_raw = len(raw)
n_kept = len(result.clean)
n_excluded = len(result.exclusions)

print(f"raw:      {n_raw}")
print(f"kept:     {n_kept}")
print(f"excluded: {n_excluded}")
print(f"kept + excluded == raw?  {n_kept + n_excluded == n_raw}")

raw:      27
kept:     11
excluded: 16
kept + excluded == raw?  True


#### Also can manually cross-check with raw data spreadsheet whether `describe_structure` missed anything:

In [3]:
describe_structure(structure, codebook)

Characters: 4
Outcomes: 5
Causal questions per outcome: 3
Attention checks: 8
  attention_1 (QID227): What did the narrator find on their brother's laptop after logging in?
  attention_2 (QID231_1): Suppose a person detonates a bomb in the middle of a community center that is filled with 
  attention_3 (QID235_1): If Sally is crushed to death by a trolley someone intentionally diverted to run them over,
  attention_4 (QID252): Which outcome below are you considering?
  attention_5 (QID263): On the questions immediately before this page, what properties of the scenario were you ra
  attention_6 (QID274): On the questions immediately before this page, which of the options below was an anchor on
  attention_7 (QID285): Which outcome below are you considering?
  attention_8 (QID296): On the questions immediately before this page, what properties of the scenario were you ra


#### Looks good for clean_survey_CHECKS_input.csv, which is actually brother_utility1_deontology1_choice1 survey data.

## Check 2 — Funnel

Print the breakdown of *why* rows were dropped, then cross-check the two
cheapest-to-verify reasons against a manual count straight from the raw
columns:

- `"not finished"` should equal the number of raw rows where `finished` is
  `False`.
- `"not sentient"` should equal the number of raw fully finished rows with a blank or
  "no"/"not sentient" `Sentience` answer.

In [4]:
print(result.exclusions["reason"].value_counts())

reason
not finished                             8
fewer than 7 attention checks correct    7
not sentient                             1
Name: count, dtype: int64


In [5]:
# Manual cross-check, independent of the cleaning code's own logic
manual_not_finished = (~raw["finished"].astype(str).str.strip().str.lower().eq("true")).sum()
print(f"manual count of unfinished rows:        {manual_not_finished}")

finished_mask = raw["finished"].astype(str).str.strip().str.lower().eq("true")
sentience_col = raw.loc[finished_mask, structure["sentience"]].fillna("").astype(str).str.strip()
manual_not_sentient = (
    (sentience_col == "") | sentience_col.str.contains(r"^no\b|not sentient", case=False, regex=True)
).sum()
print(f"manual count of bad sentience answers (among finished rows): {manual_not_sentient}")

manual count of unfinished rows:        8
manual count of bad sentience answers (among finished rows): 1


## Check 3 — Column audit

`clean_survey` only keeps columns it recognizes as part of the template. Print
what's in the raw file but *not* in the cleaned output, see if we're removing something that's needed.

In [6]:
rename_map = build_rename_map(structure)
kept_raw_qids = set(rename_map) | {"attention_correct"}

dropped = codebook[~codebook["qid"].isin(kept_raw_qids)]
print(f"{len(dropped)} raw columns are NOT in the cleaned output:\n")
print(dropped[["qid", "name"]].to_string(index=False))
    

23 raw columns are NOT in the cleaned output:

                 qid                      name
           startDate                 StartDate
             endDate                   EndDate
              status                    Status
            progress                  Progress
            finished                  Finished
        recordedDate              RecordedDate
           _recordId                ResponseId
 distributionChannel       DistributionChannel
        userLanguage              UserLanguage
    Q_RecaptchaScore          Q_RecaptchaScore
      LastSeenFlowID Last Seen Flow Element ID
   LastSeenQuestions    Last Seen Question IDs
       QID1220454155                   consent
         QID194_TEXT                      ID_q
         QID297_TEXT            other_outcomes
       QID1220597812                    demo_q
  QID1220597813_TEXT                    demo_q
       QID1220597814                    demo_q
QID1220597814_7_TEXT             demo_q_7_TEXT
             

## Check 4 — split_outputs

Check for missing rows or misclassified columns after applying the `split` function. Also
check that columns meant to appear in *both* outputs (Subject, attention, ...)
actually hold identical values in each — not just the same name.

In [7]:
util_df, cik_df = split_outputs(result.clean)

print(f"clean rows: {len(result.clean)}, util_df rows: {len(util_df)}, cik_df rows: {len(cik_df)}")
print(f"row counts match?  {len(util_df) == len(result.clean) and len(cik_df) == len(result.clean)}")

all_cols = set(result.clean.columns)
covered = set(util_df.columns) | set(cik_df.columns)
print(f"columns missing from both outputs: {sorted(all_cols - covered) or 'none'}")

cik_in_util = [c for c in util_df.columns if c.startswith("causal_") or c.startswith("outcome_prob_")]
util_in_cik = [c for c in cik_df.columns if c.startswith("char_") or c == "overall_utility"]
print(f"CIK columns misclassified into util_df: {cik_in_util or 'none'}")
print(f"utility columns misclassified into cik_df: {util_in_cik or 'none'}")

shared = sorted(set(util_df.columns) & set(cik_df.columns))
mismatched = [c for c in shared if not util_df[c].equals(cik_df[c])]
print(f"\nshared columns ({len(shared)}) with mismatched values between the two outputs: {mismatched or 'none'}")

clean rows: 11, util_df rows: 11, cik_df rows: 11
row counts match?  True
columns missing from both outputs: none
CIK columns misclassified into util_df: none
utility columns misclassified into cik_df: none

shared columns (15) with mismatched values between the two outputs: none
